### Decision tree
1) запуск алгоритма с параметрами по умолчанию и вывод некоторой статистики
2) запуск optuna-оптимизации по части гиперпараметров
3) визуализация optuna: важность параметров и контуры
4) запуск алгоритма с найденными гиперпараметрами и вывод предварительной статистики
5) сохраняем результаты дефолтного и оптимизированного алгоритма

In [27]:
from private.utils import get_reduced_mnist_data, memory_check
from public.classification_utils import DEC_TR
from public.models import ClassificationProcessor

with memory_check():
    df = get_reduced_mnist_data()
    processor = ClassificationProcessor(df, "label")   
    processor.calculate({
        DEC_TR : {},
    })
    processor.report(DEC_TR)
    processor.pick_model(DEC_TR)

decision_tree took 17.254 seconds

	decision_tree


pr_auc,roc_auc,accuracy
0.775795,0.929309,0.874357


,precision,recall,f1-score,support
0,0.922,0.929,0.926,1381.000
1,0.942,0.962,0.952,1575.000
2,0.860,0.844,0.852,1398.000
3,0.848,0.846,0.847,1428.000
4,0.864,0.873,0.869,1365.000
5,0.833,0.824,0.828,1263.000
6,0.903,0.887,0.895,1375.000
7,0.893,0.915,0.904,1459.000
8,0.835,0.813,0.824,1365.000
9,0.827,0.832,0.830,1391.000


Memory Increased by: 620.97 MB


#### запуск optuna-оптимизации по части гиперпараметров

In [29]:
from public.optuna_utils import OPT_DEC_TR, optimize

with memory_check():
    study = optimize(
        model_type=OPT_DEC_TR, 
        df=df, 
        target_column="label",
        n_trials=20
    )
    print(f"Наилучшие значения гиперпараметров {study.best_params}")
    print(f"pr_auc на обучающем наборе: {study.best_value:.2f}")

  0%|          | 0/20 [00:00<?, ?it/s]

optuna_optimize took 68.387 seconds
optuna_optimize took 82.074 seconds
optuna_optimize took 140.233 seconds
optuna_optimize took 144.420 seconds
optuna_optimize took 161.396 seconds
optuna_optimize took 173.488 seconds
optuna_optimize took 106.248 seconds
optuna_optimize took 179.876 seconds
optuna_optimize took 182.314 seconds
optuna_optimize took 186.040 seconds
optuna_optimize took 193.531 seconds
optuna_optimize took 194.273 seconds
optuna_optimize took 194.554 seconds
optuna_optimize took 129.798 seconds
optuna_optimize took 98.195 seconds
optuna_optimize took 106.899 seconds
optuna_optimize took 79.784 seconds
optuna_optimize took 75.175 seconds
optuna_optimize took 94.259 seconds
optuna_optimize took 82.408 seconds
Наилучшие значения гиперпараметров {'criterion': 'gini', 'max_depth': 28, 'min_samples_leaf': 18}
pr_auc на обучающем наборе: 0.90
Memory Increased by: -331.44 MB


#### Визуализация optuna:
1) Сравнение важности гиперпараметров
2) Отрисовка контура оптимизации. Помогает выбрать направление дальнейшей оптимизации в сторону "темных" областей

In [30]:
from optuna.visualization import plot_param_importances

plot_param_importances(study)

![Project Screenshot](./data/optuna/dec_tr_feature_importance.png)

In [31]:
from optuna.visualization import plot_contour

plot_contour(study)

![Project Screenshot](./data/optuna/dec_tr_contour.png)

#### Применение найденных лучших гиперпараметров:

In [33]:
with memory_check():
    alter_title = f'{DEC_TR}_tuned'  
    processor.calculate({
        DEC_TR : {
            'criterion': 'entropy',
            'max_depth': 30, 
            'min_samples_leaf': 19,
            'alter_title': alter_title
        },
    })
    processor.report(alter_title)
    processor.pick_model(alter_title)

decision_tree took 11.865 seconds

	decision_tree_tuned


pr_auc,roc_auc,accuracy
0.906413,0.971346,0.863643


,precision,recall,f1-score,support
0,0.895,0.936,0.915,1381.000
1,0.944,0.951,0.948,1575.000
2,0.853,0.835,0.844,1398.000
3,0.821,0.846,0.833,1428.000
4,0.852,0.855,0.853,1365.000
5,0.810,0.783,0.796,1263.000
6,0.879,0.873,0.876,1375.000
7,0.904,0.899,0.901,1459.000
8,0.851,0.803,0.826,1365.000
9,0.811,0.834,0.822,1391.000


Memory Increased by: 719.79 MB


#### Мини-репорт:

In [49]:
dec_tr = next((model for model in processor.models if model.title == DEC_TR), None)
dec_tr_tuned = next((model for model in processor.models if model.title == alter_title), None)

print(f"{DEC_TR} : {alter_title} >> {dec_tr.pr_auc} : {dec_tr_tuned.pr_auc}")

decision_tree : decision_tree_tuned >> 0.7757954221400802 : 0.9064126100316097
